In [37]:
import sys
sys.path.append('../')
from ctw.ctw import CTW

from mingpt.model import GPT
from mingpt.trainer import Trainer
import torch
from torch.utils.data import Dataset
import numpy as np
import torch
import torch.nn.functional as F
from mingpt.model import GPT
import pickle
import matplotlib.pyplot as plt

In [38]:
def markov_generate(p_stay, N):
    state = np.random.choice([0, 1])  # stationary is uniform for symmetric chain
    seq = [state]
    for _ in range(N - 1):
        if np.random.rand() < p_stay:
            seq.append(state)  # stay
        else:
            state = 1 - state  # switch
            seq.append(state)
    return np.array(seq)


# Dirichlet array generation

In [39]:
import numpy as np

def random_transition_matrix(n_states, concentration):
    alpha = np.ones(n_states) * concentration
    return np.random.dirichlet(alpha, size=n_states)  # shape (n_states, n_states)

def finite_state_generate(transition_matrix, emission_probs, N):
    n_states = len(emission_probs)
    state = np.random.choice(n_states)
    seq = []
    for _ in range(N):
        bit = 1 if np.random.rand() < emission_probs[state] else 0
        seq.append(bit)
        state = np.random.choice(n_states, p=transition_matrix[state])
    return np.array(seq)



In [40]:
class FixedFSMDataset(Dataset):
    def __init__(self, emission_probs, n=500, num_samples=10000, n_states=4, concentration=0.1):
        self.block_size = n - 1
        self.data = []
        for _ in range(num_samples):
            T = random_transition_matrix(n_states, concentration)
            seq = torch.tensor(finite_state_generate(T, emission_probs, n), dtype=torch.long)
            x = seq[:-1].clone()
            y = seq[1:].clone()
            self.data.append((x, y))

    def get_block_size(self):
        return self.block_size

    def __getitem__(self, idx):
        return self.data[idx]

    def __len__(self):
        return len(self.data)

In [41]:
# Model configure

model_save_folder = '../models/'

def model_configure(dataset, iters=1000, learn_rate=1e-4):
    model_config = GPT.get_default_config()
    model_config.model_type = 'gpt-nano'
    model_config.vocab_size = 2
    model_config.block_size = dataset.get_block_size()
    model = GPT(model_config)

    device = 'mps'
    model = model.to(device)

    train_config = Trainer.get_default_config()
    train_config.learning_rate = learn_rate
    train_config.max_iters = iters
    train_config.num_workers = 0
    train_config.device = 'mps'
    trainer = Trainer(train_config, model, dataset)
    return model, trainer

def batch_end_callback(trainer):
    if trainer.iter_num % 100 == 0:
        print(f"iter_dt {trainer.iter_num}; iter {trainer.iter_num}: train loss {trainer.loss.item():.5f}")

def train_run(model, trainer, output_name):
    save_dir = model_save_folder + output_name
    print(f'Saving to {save_dir}')
    trainer.set_callback('on_batch_end', batch_end_callback)
    trainer.run()
    torch.save({
        'state_dict': model.state_dict(),
        'block_size': model.block_size
    }, save_dir)
    print(f'Saved to {save_dir}')

# Experiment infrastructure

In [42]:
experiment_save_folder = '../experiments/'
N_values = [10, 25, 50, 100, 200, 499]

In [43]:

def iid_generate(p_dict, N):
    p0 = p_dict[0]
    p1 = p_dict[1]
    return np.random.choice([0,1], size=N, p=[p0, p1])

def generate_bin_pmf():
    p = np.random.rand()
    dict_out = {
        0: p,
        1: 1-p
    }
    return dict_out

def convert_sample_to_pmf(sample):
    values, counts = np.unique(sample, return_counts=True)
    probs = counts/counts.sum()
    return dict(zip(values, probs))

def entropy_calc(pmf):
    pmf = clean_pmf(pmf)
    pmf_array = np.asarray(list(pmf.values()))
    return -np.sum(pmf_array * np.log2(pmf_array))

def clean_pmf(pmf):
    return {k: v for k, v in pmf.items() if v > 0.0}

# CTW prediction machine

In [ ]:

def ctw_p_array(sequence, depth=8):
    sequence = [int(x) for x in sequence]  
    ctw = CTW(depth=depth, symbols=2)
    distributions = ctw.predict_sequence(sequence)
    p_ones = distributions[1, :].tolist()
    p_array = [0.5] * depth + p_ones
    return p_array

p = generate_bin_pmf()
seq = iid_generate(p, 20)
print(ctw_p_array(seq))

[0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.625, 0.5833333333333333, 0.5, 0.346875, 0.3880083732057416, 0.6545168580503299, 0.20779139350955367, 0.23818212397133823, 0.8017227073038989, 0.13704061287152244, 0.1497187077668784]


In [45]:
def sequential_universal_source_coding(seq, p_array = None):
    total_bits = 0
    k = 0
    i = 0

    for bit in seq:
        if p_array is not None:
            p1 = p_array[i]
        else:
            p1 = (k+1) / (i+2)
        p0 = 1-p1

        if bit == 1:
            total_bits += -np.log2(p1)
        else:
            total_bits += -np.log2(p0)
        k += bit
        i += 1

    bits_per_symbol = total_bits / len(seq)
    return bits_per_symbol

In [46]:

# generate an array of probabilities given a bit sequence

def transformer_p_array(sequence, model, device='mps'):
    model.eval()
    sequence=torch.tensor(sequence, dtype=torch.long, device=device).unsqueeze(0)

    with torch.no_grad():
        logits, _ = model(sequence)  # [1, T, vocab_size]
        # convert to probabilities
        probs = F.softmax(logits, dim=-1)  # [1, T, vocab_size]
        # extract probability of token=1 at each timestep
        p_array = [0.5] + probs[0, :-1, 1].tolist()
    
    return p_array

In [47]:
def load_model(model_dir):
    checkpoint = torch.load(model_dir, map_location='cpu')
    block_size = checkpoint['block_size']
    
    model_config = GPT.get_default_config()
    model_config.model_type = 'gpt-nano'
    model_config.vocab_size = 2
    model_config.block_size = block_size
    
    model = GPT(model_config)
    model.load_state_dict(checkpoint['state_dict'])
    model = model.to('mps')
    model.eval()
    return model

In [48]:
def run_fsm_experiment(model, alpha, n_states, emission_probs, N_values, output_file, n_trials=100, device='mps', ctw_depth=8):
    all_results = {alpha: {}}
    save_dir = experiment_save_folder + output_file
    T = random_transition_matrix(n_states, alpha)
    
    for N in N_values:
        laplace_bps_trials = []
        transformer_bps_trials = []
        ctw_bps_trials = []
        
        for trial in range(n_trials):
            seq = finite_state_generate(T, emission_probs, N)
            
            laplace_bps_trials.append(sequential_universal_source_coding(seq))
            
            p_array = transformer_p_array(seq, model, device=device)
            transformer_bps_trials.append(sequential_universal_source_coding(seq, p_array=p_array))
            
            p_array_ctw = ctw_p_array(seq, depth=ctw_depth)
            ctw_bps_trials.append(sequential_universal_source_coding(seq, p_array=p_array_ctw))
        
        all_results[alpha][N] = {
            'laplace': (np.mean(laplace_bps_trials), np.std(laplace_bps_trials)),
            'transformer': (np.mean(transformer_bps_trials), np.std(transformer_bps_trials)),
            'ctw': (np.mean(ctw_bps_trials), np.std(ctw_bps_trials)),
        }
        print(f"  N={N}: laplace={np.mean(laplace_bps_trials):.4f}, transformer={np.mean(transformer_bps_trials):.4f}, ctw={np.mean(ctw_bps_trials):.4f}")
    
    with open(save_dir, 'wb') as f:
        pickle.dump(all_results, f)
    print(f'Saved to {save_dir}')

In [49]:
N_values = [10, 25, 50, 100, 200, 300, 499]
n_trials = 100
seq_samples = 500
num_samples = 10000
iters = 500
emission_probs = [0.1, 0.35, 0.65, 0.9]
alpha_sweep = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
n_states = 4

for alpha in alpha_sweep:
    dataset = FixedFSMDataset(emission_probs=emission_probs, n=seq_samples, num_samples=num_samples, n_states=n_states, concentration=alpha)
    model, trainer = model_configure(dataset, iters=iters)
    train_run(model, trainer, f'fsm_{alpha}_model.pt')
    run_fsm_experiment(model, alpha, n_states, emission_probs, N_values, output_file=f'fsm_{alpha}_experiment.pkl', n_trials=n_trials, device='mps')


number of parameters: 0.11M
running on device mps
Saving to ../models/fsm_0.1_model.pt
iter_dt 0; iter 0: train loss 0.69297
iter_dt 100; iter 100: train loss 0.61383
iter_dt 200; iter 200: train loss 0.59710
iter_dt 300; iter 300: train loss 0.59266
iter_dt 400; iter 400: train loss 0.58524
Saved to ../models/fsm_0.1_model.pt
  N=10: laplace=1.0393, transformer=1.0122, ctw=1.0053
  N=25: laplace=1.0026, transformer=0.9922, ctw=1.0247
  N=50: laplace=0.9791, transformer=0.9782, ctw=1.0039
  N=100: laplace=0.9661, transformer=0.9687, ctw=0.9801
  N=200: laplace=0.9696, transformer=0.9716, ctw=0.9767
  N=300: laplace=0.9703, transformer=0.9717, ctw=0.9751
  N=499: laplace=0.9783, transformer=0.9813, ctw=0.9797
Saved to ../experiments/fsm_0.1_experiment.pkl
number of parameters: 0.11M
running on device mps
Saving to ../models/fsm_0.5_model.pt
iter_dt 0; iter 0: train loss 0.69600
iter_dt 100; iter 100: train loss 0.67453
iter_dt 200; iter 200: train loss 0.67247
iter_dt 300; iter 300: tra